# Indoor Coverage Map Predictor
## Machine Learning for Indoor Cellular Coverage Prediction Using Ray Tracing

I use this notebook to predict indoor coverage for any room I specify — without rerunning training.
The trained U-Net model I built in my main research notebook is loaded directly from Google Drive.

**How to use this notebook in 3 steps:**
1. **Run Cell 1 (Setup)** — mounts Drive and loads my trained models
2. **Edit Cell 2 (Define My Room)** — describe my room geometry and transmitter
3. **Run Cells 3–5** — get coverage maps and statistics instantly

---
**Supported wall materials:** `concrete` · `drywall` · `brick` · `thick_concrete` · `reinforced`  
**Window material:** `glass`  
**Furniture materials:** `wood` · `metal`  
**Frequencies:** `700e6` (700 MHz) or `3.5e9` (3.5 GHz)

## Cell 1 — Setup: Mount Drive and Load My Trained Models

In [ ]:
# ==============================================================================
# CONFIGURE PATHS — update these if your Drive folder is different
# ==============================================================================
MODEL_DIR    = '/content/drive/MyDrive/part4_outputs'               # where coverage_unet.keras is saved
METADATA_DIR = '/content/drive/MyDrive/dataset/dataset_processed'   # where representation_metadata.json is saved
# ==============================================================================

!pip install -q numpy matplotlib

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap, BoundaryNorm
import json, time
from dataclasses import dataclass
from typing import List
from pathlib import Path

import tensorflow as tf
from tensorflow import keras

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# ── Verify files exist ────────────────────────────────────────────────────────
required = [
    Path(MODEL_DIR) / 'coverage_unet.keras',
    Path(METADATA_DIR) / 'representation_metadata.json',
]
for p in required:
    print(f'  {"✓" if p.exists() else "✗  MISSING"}  {p}')

# ── Load metadata ─────────────────────────────────────────────────────────────
with open(Path(METADATA_DIR) / 'representation_metadata.json') as f:
    meta = json.load(f)

SIGNAL_MEAN       = meta['signal_mean_dbm']
SIGNAL_STD        = meta['signal_std_dbm']
GRID_W            = meta['grid_w']
GRID_H            = meta['grid_h']
GRID_RESOLUTION   = meta['grid_resolution_m']
RECEIVER_HEIGHTS  = meta['receiver_heights_m']
NUM_HEIGHT_LEVELS = meta['num_height_levels']
FREQ_MIN          = min(meta['frequencies_hz'])
FREQ_MAX          = max(meta['frequencies_hz'])

# Coverage thresholds and class names (must match training)
COVERAGE_THRESHOLDS  = [-154, -60, -50, -43]   # dBm
COVERAGE_CLASS_NAMES = ['No Signal', 'Poor', 'Fair', 'Good', 'Excellent']
NUM_CLASSES = len(COVERAGE_CLASS_NAMES)

# Material encoding (must match Part 3)
MATERIAL_TO_IDX = {
    'concrete': 0, 'drywall': 1, 'brick': 2,
    'thick_concrete': 3, 'reinforced': 4,
    'glass': 5, 'wood': 6, 'metal': 7,
}

# ── Custom layer needed to reload U-Net ──────────────────────────────────────
def crop_to_match(inputs):
    """Crop upsampled tensor to match skip-connection tensor size."""
    to_crop, target = inputs
    tc = tf.shape(to_crop); th = tf.shape(target)
    dh = (tc[1] - th[1]) // 2; dw = (tc[2] - th[2]) // 2
    return to_crop[:, dh:dh + th[1], dw:dw + th[2], :]

# ── Load regression U-Net ─────────────────────────────────────────────────────
print('\nLoading regression U-Net...')
reg_model = keras.models.load_model(
    str(Path(MODEL_DIR) / 'coverage_unet.keras'),
    custom_objects={'crop_to_match': crop_to_match}
)
print(f'  Parameters: {reg_model.count_params():,}')

# ── Load classifier U-Net if available ───────────────────────────────────────
clf_model = None
clf_path  = Path(MODEL_DIR) / 'coverage_clf_unet.keras'
if clf_path.exists():
    print('Loading classifier U-Net...')
    def crop_to_match_clf(inputs):
        return crop_to_match(inputs)
    clf_model = keras.models.load_model(
        str(clf_path),
        custom_objects={'crop_to_match_clf': crop_to_match_clf,
                        'crop_to_match': crop_to_match}
    )
    print(f'  Parameters: {clf_model.count_params():,}')
else:
    print('  Classifier not found — coverage classes derived from regression output.')

print(f'\n✓ Ready.  Grid: {GRID_H} rows x {GRID_W} cols  ({GRID_RESOLUTION} m/cell)')
print(f'  Normalisation: mean={SIGNAL_MEAN:.2f} dBm   std={SIGNAL_STD:.2f} dBm')

## Cell 2 — ✏️ Define My Room

I edit this cell to describe the room I want to predict coverage for, then run all cells below.

**Coordinate system:** X increases to the right, Y increases upward. All measurements in **metres**.

In [ ]:
# ==============================================================================
#  ✏️  EDIT THIS CELL — describe my room
# ==============================================================================

from dataclasses import dataclass
from typing import List
import numpy as np

@dataclass
class Point:
    """2D point in floorplan coordinates (x, y)"""
    x: float
    y: float

@dataclass
class Wall:
    """Wall defined by start and end points"""
    start: Point
    end: Point
    material: str
    thickness: float = 0.15

@dataclass
class Window:
    """Window on a wall"""
    wall_index: int       # index into walls list
    start_offset: float   # metres from wall start
    length: float
    material: str = 'glass'

@dataclass
class Door:
    """Door on a wall"""
    wall_index: int
    start_offset: float
    length: float
    material: str = 'wood'

@dataclass
class Furniture:
    """Furniture as axis-aligned bounding box"""
    bottom_left: Point
    width: float
    depth: float
    material: str
    height: float = 0.75

@dataclass
class Floorplan:
    """Complete floorplan description"""
    walls: List[Wall]
    windows: List[Window]
    doors: List[Door]
    furniture: List[Furniture]
    name: str = 'my_room'

# ==============================================================================
# EXAMPLE: 8 m x 6 m room with an interior drywall partition
# Change these values to match your room
# ==============================================================================

my_walls = [
    # Exterior walls
    Wall(Point(0, 0), Point(8, 0), 'concrete', thickness=0.20),   # wall 0 - South
    Wall(Point(8, 0), Point(8, 6), 'concrete', thickness=0.20),   # wall 1 - East
    Wall(Point(8, 6), Point(0, 6), 'concrete', thickness=0.20),   # wall 2 - North
    Wall(Point(0, 6), Point(0, 0), 'concrete', thickness=0.20),   # wall 3 - West
    # Interior partition
    Wall(Point(4, 0), Point(4, 6), 'drywall',  thickness=0.10),   # wall 4 - Interior
]

my_windows = [
    Window(wall_index=0, start_offset=1.5, length=2.0),   # South wall window
    Window(wall_index=1, start_offset=2.0, length=1.5),   # East wall window
]

my_doors = [
    Door(wall_index=4, start_offset=2.5, length=0.9),     # Interior wall door
]

my_furniture = [
    Furniture(Point(0.5, 0.5), width=2.0, depth=1.0, material='wood'),
    Furniture(Point(0.5, 4.0), width=1.0, depth=1.5, material='wood'),
    Furniture(Point(5.0, 1.0), width=1.5, depth=0.8, material='metal'),
    Furniture(Point(5.5, 3.5), width=2.0, depth=1.5, material='wood'),
]

# ── Transmitter ───────────────────────────────────────────────────────────────
TX_POSITION  = np.array([-12.0, 3.0, 2.5])   # [x, y, z] in metres
TX_FREQUENCY = 3.5e9                          # 700e6 or 3.5e9
TX_POWER_DBM = 30.0                           # dBm

# ── Assemble ──────────────────────────────────────────────────────────────────
my_floorplan = Floorplan(
    walls=my_walls, windows=my_windows,
    doors=my_doors, furniture=my_furniture,
    name='my_room'
)

xs = [w.start.x for w in my_walls] + [w.end.x for w in my_walls]
ys = [w.start.y for w in my_walls] + [w.end.y for w in my_walls]
print('Room spec loaded:')
print(f'  Bounding box : {max(xs)-min(xs):.1f} m x {max(ys)-min(ys):.1f} m')
print(f'  Walls: {len(my_walls)}  Windows: {len(my_windows)}  Doors: {len(my_doors)}  Furniture: {len(my_furniture)}')
print(f'  TX: {TX_POSITION}  {TX_FREQUENCY/1e9:.2f} GHz  {TX_POWER_DBM} dBm')

## Cell 3 — Build Input Tensor and Run Prediction

In [ ]:
def rasterize_line(x0, y0, x1, y1, gw, gh, res):
    """Return (row, col) pairs that lie on a line segment."""
    n    = max(2, int(np.sqrt((x1-x0)**2+(y1-y0)**2)/res*10))
    ts   = np.linspace(0, 1, n)
    cols = np.clip(((x0+ts*(x1-x0))/res).astype(int), 0, gw-1)
    rows = np.clip(((y0+ts*(y1-y0))/res).astype(int), 0, gh-1)
    return list(set(zip(rows.tolist(), cols.tolist())))


def build_input_tensor(fp, tx_pos, tx_freq, gw, gh, res):
    """
    Encode a floorplan + transmitter into a 6-channel input tensor.
    Channels: wall presence, wall material, window, furniture, TX distance, frequency.
    """
    xs    = [w.start.x for w in fp.walls]+[w.end.x for w in fp.walls]
    ys    = [w.start.y for w in fp.walls]+[w.end.y for w in fp.walls]
    x_min = min(xs); y_min = min(ys)
    T     = np.zeros((gh, gw, 6), dtype=np.float32)

    for wall in fp.walls:
        mi = MATERIAL_TO_IDX.get(wall.material, 0) / 7.0
        for r, c in rasterize_line(wall.start.x-x_min, wall.start.y-y_min,
                                    wall.end.x-x_min,   wall.end.y-y_min, gw, gh, res):
            T[r, c, 0] = 1.0
            T[r, c, 1] = mi

    for win in fp.windows:
        if win.wall_index >= len(fp.walls): continue
        w  = fp.walls[win.wall_index]
        wl = np.sqrt((w.end.x-w.start.x)**2+(w.end.y-w.start.y)**2)
        if wl < 1e-6: continue
        ux=(w.end.x-w.start.x)/wl; uy=(w.end.y-w.start.y)/wl
        wx0=w.start.x+ux*win.start_offset; wy0=w.start.y+uy*win.start_offset
        for r, c in rasterize_line(wx0-x_min, wy0-y_min,
                                    wx0+ux*win.length-x_min, wy0+uy*win.length-y_min,
                                    gw, gh, res):
            T[r, c, 2] = 1.0

    for furn in fp.furniture:
        c0=int(np.clip((furn.bottom_left.x-x_min)/res,0,gw-1))
        r0=int(np.clip((furn.bottom_left.y-y_min)/res,0,gh-1))
        c1=int(np.clip((furn.bottom_left.x+furn.width-x_min)/res,0,gw-1))
        r1=int(np.clip((furn.bottom_left.y+furn.depth-y_min)/res,0,gh-1))
        T[r0:r1+1, c0:c1+1, 3] = 1.0

    tx_x = tx_pos[0]-x_min; tx_y = tx_pos[1]-y_min
    cg, rg = np.meshgrid(np.arange(gw)*res, np.arange(gh)*res)
    dist   = np.sqrt((cg-tx_x)**2+(rg-tx_y)**2).astype(np.float32)
    T[:,:,4] = dist/(dist.max() if dist.max()>1e-6 else 1.0)
    T[:,:,5] = float((tx_freq-FREQ_MIN)/(FREQ_MAX-FREQ_MIN))

    return T[np.newaxis]   # (1, H, W, 6)


def dbm_to_class(dbm):
    cls = np.zeros_like(dbm, dtype=np.int32)
    for i, thresh in enumerate(COVERAGE_THRESHOLDS):
        cls[dbm >= thresh] = i + 1
    return np.clip(cls, 0, NUM_CLASSES - 1)


# ── Build tensor and predict ──────────────────────────────────────────────────
print('Building input tensor...')
X_user = build_input_tensor(
    my_floorplan, TX_POSITION, TX_FREQUENCY, GRID_W, GRID_H, GRID_RESOLUTION
)

print('Running regression U-Net...')
t0    = time.time()
y_norm = reg_model.predict(X_user, verbose=0)
y_dbm  = y_norm[0] * SIGNAL_STD + SIGNAL_MEAN   # (H, W, num_heights)

if clf_model is not None:
    print('Running classifier U-Net...')
    logits_flat = clf_model.predict(X_user, verbose=0)[0]
    H, W = y_dbm.shape[:2]
    logits_r   = logits_flat.reshape(H, W, NUM_HEIGHT_LEVELS, NUM_CLASSES)
    y_cls_pred = np.argmax(logits_r, axis=-1)
else:
    y_cls_pred = dbm_to_class(y_dbm)

xs    = [w.start.x for w in my_walls]+[w.end.x for w in my_walls]
ys    = [w.start.y for w in my_walls]+[w.end.y for w in my_walls]
x_min, x_max = min(xs), max(xs)
y_min, y_max = min(ys), max(ys)

print(f'Done in {time.time()-t0:.1f}s')
print(f'Signal range at 1.5 m: {y_dbm[:,:,1].min():.1f} to {y_dbm[:,:,1].max():.1f} dBm')

## Cell 4 — Visualise My Coverage Maps

In [ ]:
MAT_COLORS = {
    'concrete': '#78909c', 'thick_concrete': '#546e7a', 'reinforced': '#37474f',
    'brick': '#e57373',    'drywall': '#fff176',         'glass': '#80deea',
    'wood': '#a5d6a7',     'metal': '#90caf9',
}
CLASS_COLORS = ['#d32f2f', '#f57c00', '#fdd835', '#388e3c', '#1565c0']
cmap_cls = ListedColormap(CLASS_COLORS)
norm_cls = BoundaryNorm(np.arange(-0.5, NUM_CLASSES, 1), cmap_cls.N)


def overlay_floorplan(ax, fp, x0, y0):
    for w in fp.walls:
        ax.plot([w.start.x-x0, w.end.x-x0], [w.start.y-y0, w.end.y-y0],
                color=MAT_COLORS.get(w.material,'#aaa'), lw=4, zorder=5)
    for win in fp.windows:
        if win.wall_index >= len(fp.walls): continue
        wall=fp.walls[win.wall_index]
        wl=np.sqrt((wall.end.x-wall.start.x)**2+(wall.end.y-wall.start.y)**2)
        if wl<1e-6: continue
        ux=(wall.end.x-wall.start.x)/wl; uy=(wall.end.y-wall.start.y)/wl
        wx0=wall.start.x+ux*win.start_offset; wy0=wall.start.y+uy*win.start_offset
        ax.plot([wx0-x0, wx0+ux*win.length-x0],[wy0-y0, wy0+uy*win.length-y0],
                color='#00e5ff', lw=6, zorder=6, solid_capstyle='butt')
    for f in fp.furniture:
        ax.add_patch(plt.Rectangle(
            (f.bottom_left.x-x0, f.bottom_left.y-y0), f.width, f.depth,
            fc=MAT_COLORS.get(f.material,'#ccc'), ec='k', lw=0.7, alpha=0.7, zorder=7))


hlabels = ['0.5 m (floor level)', '1.5 m (phone height)', '2.5 m (elevated)']
tx_plot = [TX_POSITION[0]-x_min, TX_POSITION[1]-y_min]
vmin    = float(np.percentile(y_dbm, 2))
vmax    = float(np.percentile(y_dbm, 98))
ext     = [0, y_dbm.shape[1]*GRID_RESOLUTION, 0, y_dbm.shape[0]*GRID_RESOLUTION]

fig, axes = plt.subplots(3, 2, figsize=(14, 16),
                          gridspec_kw={'hspace': 0.45, 'wspace': 0.3})
fig.suptitle(
    f'Predicted Indoor Coverage — "{my_floorplan.name}"\n'
    f'TX: {TX_POSITION}  |  {TX_FREQUENCY/1e9:.2f} GHz  |  {TX_POWER_DBM} dBm',
    fontsize=13, fontweight='bold'
)

for h in range(3):
    dbm = y_dbm[:, :, h]
    cls = y_cls_pred[:, :, h]
    ax_h, ax_c = axes[h][0], axes[h][1]

    im = ax_h.imshow(dbm, origin='lower', cmap='RdYlGn',
                     vmin=vmin, vmax=vmax, extent=ext, interpolation='bilinear')
    overlay_floorplan(ax_h, my_floorplan, x_min, y_min)
    ax_h.plot(*tx_plot, 'r*', ms=14, zorder=10, label='TX')
    ax_h.set_title(f'Signal Strength — {hlabels[h]}', fontsize=10)
    ax_h.set_xlabel('X (m)'); ax_h.set_ylabel('Y (m)')
    plt.colorbar(im, ax=ax_h, label='dBm')
    ax_h.legend(loc='upper right', fontsize=8)

    ax_c.imshow(cls, origin='lower', cmap=cmap_cls, norm=norm_cls,
                extent=ext, interpolation='nearest')
    overlay_floorplan(ax_c, my_floorplan, x_min, y_min)
    ax_c.plot(*tx_plot, 'r*', ms=14, zorder=10)
    ax_c.set_title(f'Coverage Class — {hlabels[h]}', fontsize=10)
    ax_c.set_xlabel('X (m)'); ax_c.set_ylabel('Y (m)')
    patches = [mpatches.Patch(color=CLASS_COLORS[i], label=COVERAGE_CLASS_NAMES[i])
               for i in range(NUM_CLASSES)]
    ax_c.legend(handles=patches, loc='upper right', fontsize=6, ncol=2)

plt.tight_layout(rect=[0, 0, 1, 0.96])
out_png = f'/content/coverage_{my_floorplan.name}.png'
plt.savefig(out_png, dpi=130, bbox_inches='tight')
plt.show()
print(f'Saved: {out_png}')

## Cell 5 — Coverage Statistics Summary

In [ ]:
print('=' * 68)
print(f'COVERAGE SUMMARY — {my_floorplan.name}')
print('=' * 68)
print(f'Room size  : {x_max-x_min:.1f} m x {y_max-y_min:.1f} m')
print(f'TX         : {TX_POSITION}   {TX_FREQUENCY/1e9:.2f} GHz   {TX_POWER_DBM} dBm')
print()

for h, hl in enumerate(['0.5 m (floor)', '1.5 m (phone)', '2.5 m (elevated)']):
    dbm   = y_dbm[:, :, h]
    cls   = y_cls_pred[:, :, h]
    total = dbm.size
    print(f'Height {h} — {hl}')
    print(f'  Signal: {dbm.min():.1f} to {dbm.max():.1f} dBm   (mean {dbm.mean():.1f} dBm)')
    for c, name in enumerate(COVERAGE_CLASS_NAMES):
        n   = int((cls == c).sum())
        pct = 100 * n / total
        bar = chr(9608) * int(pct / 2)
        print(f'  {name:<12}: {n:6d} cells ({pct:5.1f}%)  {bar}')
    print()

print('-' * 68)
print('Tip: edit Cell 2 and rerun Cells 3-5 to test different rooms or')
print('transmitter positions — no retraining needed!')